### This demo showcases the implementation of story RSPY-808 (Implement STAC view of EDRS sessions)
See https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-808


In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *
import pprint
pp = pprint.PrettyPrinter(indent=2, width=80, sort_dicts=False, compact=True)

auxip_client, cadip_client, catalog_client, staging_client, prip_client, edrs_client = init_demo()

In [ ]:
# STAC API landing page. /edrs/
edrs_client.get_landing()

In [ ]:
# STAC collections the user has permission to access. '/edrs/collections'
collections = edrs_client.get_collections()
for c in collections:
    pp.pprint(c.to_dict()['id'])

In [ ]:
# Queryable fields. '/edrs/queryables'
general_queryables = edrs_client.get_queryables()
assert isinstance(general_queryables, dict)
pprint.pp(general_queryables)

In [ ]:
fields = list(general_queryables["properties"].keys())
print(fields)

In [ ]:
collection_queryables = edrs_client.get_collection_queryables(collection_id='s1_pedc')
assert isinstance(collection_queryables, dict)
fields = list(collection_queryables["properties"].keys())
print(fields)

In [ ]:
# STAC collections the user has permission to access. '/edrs/collections/{collectionId}'
edrs_client.get_collection(collection_id='s1_pedc')

In [ ]:
# STAC collections the user has permission to access. '/edrs/collections/{collectionId}'
edrs_client.get_collection(collection_id='s2_pedc')

In [ ]:
#'/edrs/collections/{collectionId}/items'
items_list = list(edrs_client.get_items(collection_id="s1_pedc"))
pp.pprint(items_list)

In [ ]:
#'/edrs/collections/{collectionId}/items/{featureId}'
edrs_client.get_item(collection_id='s1_pedc', item_id="DCS_02_202502131123000000987654")

In [ ]:
#'/edrs/collections/{collectionId}/items/{featureId}'
edrs_client.get_item(collection_id='s1_pedc', item_id="DCS_01_202501270945000000112233")

In [ ]:
items_iter = edrs_client.get_items(collection_id="s1_pedc", sortby='+published', limit=1, page=1)
items_list = list(items_iter)
assert len(items_list) == 1
pp.pprint(items_list)
items_iter = edrs_client.get_items(collection_id="s1_pedc", sortby='+published', limit=1, page=2)
items_list = list(items_iter)
assert len(items_list) == 1
pp.pprint(items_list)

In [ ]:
items_iter = edrs_client.get_items(collection_id="s1_pedc", sortby='+published', limit=2, page=1)
items_list = list(items_iter)
assert len(items_list) == 2
pp.pprint(items_list)
items_iter = edrs_client.get_items(collection_id="s1_pedc", sortby='+published', limit=2, page=2)
items_list = list(items_iter)
assert not items_list
pp.pprint(items_list)

In [ ]:
items = edrs_client.get_items(collection_id='s1_pedc', filter="platform='sentinel-1c'")
items_list = list(items)
assert len(items_list) == 1
pp.pprint(items_list)

In [ ]:
items = edrs_client.get_items(collection_id='s1_pedc', filter="constellation='sentinel-1'")
items_list = list(items)
assert len(items_list) == 2
pp.pprint(items_list)
#import json
#print(json.dumps(items_list[0].to_dict(), indent=2))

In [ ]:
# datetime filters
items = edrs_client.get_items(collection_id="s1_pedc", datetime="2026-01-01T00:00:00Z/..")
items_list = list(items)
assert not items_list
items_list

In [ ]:
items = edrs_client.get_items(collection_id="s1_pedc", datetime="2024-01-01T00:00:00Z/..")
items_list = list(items)
assert len(items_list) == 2
items_list

In [ ]:
items = edrs_client.get_items(collection_id="s1_pedc", datetime="2024-02-13T11:23:00Z/2025-02-13T11:33:00Z")
items_list = list(items)
assert len(items_list) == 2
items_list

In [ ]:
items = edrs_client.get_items(collection_id="s1_pedc", datetime="../2025-02-13T12:00:00Z")
items_list = list(items)
assert len(items_list) == 2
items_list

In [ ]:
items = edrs_client.get_items(collection_id="s1_pedc", datetime="2024-01-01T00:00:00Z/..")
items_list = list(items)
assert len(items_list) == 2
items_list

In [ ]:
items = edrs_client.get_items(
    collection_id="s1_pedc",
    filter="published='2025-02-13T11:28:42Z'",
)
print(list(items))

items = edrs_client.get_items(
    collection_id="s2_pedc",
    filter="start_datetime='2025-04-19T12:37:00.000Z'",
)
print(list(items))

items = edrs_client.get_items(
    collection_id="s1_pedc",
    filter="end_datetime='2024-04-10T08:42:14Z'",
)
print(list(items))

In [ ]:
items_iter = edrs_client.get_items(
    collection_id='s1_pedc',
    filter="platform='sentinel-1c' AND constellation='sentinel-1'",
    sortby='-published',
    limit=2,
    page=1,
)
assert len(items_list) == 1
items_list

In [ ]:
# cql2-json filter
import json
items_iter = edrs_client.get_items(
    collection_id="s1_pedc",
    **{
        "filter-lang": "cql2-json",
        "filter": json.dumps({
            "op": "=",
            "args": [
                {"property": "published"},
                {"literal": "2025-02-13T11:28:42Z"},
            ],
        }),
    },
)
items_list = list(items_iter)
assert len(items_list) == 1
items_list

In [ ]:
items_iter = edrs_client.get_items(
    collection_id="s1_pedc",
    **{
        "filter-lang": "cql2-json",
        "filter": json.dumps({
            "op": "and",
            "args": [
                {
                    "op": "=",
                    "args": [
                        {"property": "platform"},
                        {"literal": "sentinel-1c"},
                    ],
                },
                {
                    "op": "=",
                    "args": [
                        {"property": "constellation"},
                        {"literal": "sentinel-1"},
                    ],
                },
            ],
        }),
    },
)
items_list = list(items_iter)
assert len(items_list) == 1
items_list

In [ ]:
## STAGING
items_list_s1_pedc = list(edrs_client.get_items(collection_id="s1_pedc"))
items_list_s2_pedc = list(edrs_client.get_items(collection_id="s2_pedc"))
items_to_stage = [items_list_s1_pedc, items_list_s2_pedc]
items_to_stage

In [ ]:
# Create a test collection 
CATALOG_COLLECTION_ID = "SPRINT30_EDRS_TEST_COLLECTION"
collection = create_test_collection(CATALOG_COLLECTION_ID)
items = catalog_client.get_items(CATALOG_COLLECTION_ID)
list(items)

In [ ]:
init_dask_cluster_staging(scale=2)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

pp = pprint.PrettyPrinter(indent=2, width=80, sort_dicts=False, compact=True)

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_CORES_EOPF=4 docker compose up # ...
display(dask_cluster_staging)

In [ ]:
staging_resp_list = []
for items in items_list:
    staging_resp_list.append(staging_client.run_staging(items.to_dict(), CATALOG_COLLECTION_ID))

for resp in staging_resp_list:
    staging_client.wait_for_jobs(resp, logger)

In [ ]:
# Check that each of the job previously launched are successful
for entry in staging_resp_list:
    inner = next(iter(entry.values()))
    job_id = inner["jobID"]
    job_results = staging_client.get_job_results(job_id)
    print(f"Results from job {job_id}: {job_results}")
    assert job_results == "successful"

In [ ]:
result = list(catalog_client.get_collection(CATALOG_COLLECTION_ID).get_items())
result

In [ ]:
# DELETE THE WHOLE COLLECTION
result = catalog_client.remove_collection(CATALOG_COLLECTION_ID)
assert result.json()["deleted collection"] == CATALOG_COLLECTION_ID
pp.pprint(result.json())

In [ ]:
shutdown = False
if shutdown:    
    # shutdown the clusters
    shutdown_dask_clusters(dask_gateway_staging, dask_cluster_staging.name)
 
# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.